# 🧩 Mini-Lab: Define the AI Protocol

**Module: The Replaceable Model Interface** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Define** a typed `Protocol` class that any AI backend must satisfy
2. **Understand** how dependency injection lets you swap backends without changing business logic
3. **Implement** a mock backend and a simple backend that both conform to the same interface
4. **Recognize** how type hints and dataclasses make the interface explicit and self-documenting

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Interface Seam | A small typed contract between your app and an AI vendor. Your adapter returns your dataclass, never their SDK response object, so providers can be swapped without rewrites. |
| Dependency Injection | Pass the backend in, callers choose which model client to use, while your pipeline code stays unchanged and testable. |
| Type Hints | Type hints are optional labels on variables and functions that describe what kinds of values you expect, helping tools catch bugs and making code easier to read and refactor. |
| Dataclass Shape | Use a function for behavior and a dataclass for shape: a named, typed bundle of fields that makes pipeline inputs and outputs explicit. Use a class only when you truly need both state and behavior together. |


## Recap

In **mini-seam-teardown**, we identified the problem: vendor-specific code (imports, model names, response parsing) was tangled with business logic. We drew a ✂ seam line showing *where* to cut.

Now we'll **build what goes at that seam** — a small typed interface (a Python `Protocol`) and the dependency-injection pattern to wire it up.

## Step 1 — Define the Data Shapes

Before defining the interface, we need to agree on **what goes in** and **what comes out**. Dataclasses give us lightweight, typed containers for this.

In [1]:
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class AIRequest:
    """What the business logic sends to the AI backend."""
    prompt: str
    system: str = "You are a helpful assistant."
    temperature: float = 0.0
    max_tokens: int = 256


@dataclass
class AIResponse:
    """What the AI backend returns to the business logic."""
    text: str
    model: str = "unknown"
    usage: dict = field(default_factory=dict)


# Quick test: create an instance to confirm the shape
req = AIRequest(prompt="Hello!", temperature=0.5)
print(f"AIRequest  → prompt={req.prompt!r}, temp={req.temperature}, max_tokens={req.max_tokens}")

resp = AIResponse(text="Hi there!", model="mock-1.0")
print(f"AIResponse → text={resp.text!r}, model={resp.model}")

AIRequest  → prompt='Hello!', temp=0.5, max_tokens=256
AIResponse → text='Hi there!', model=mock-1.0


**Why dataclasses?**  
- They are self-documenting: field names and types are visible at a glance.  
- They enforce a consistent shape — every backend receives the same `AIRequest` and returns the same `AIResponse`.  
- Default values (like `temperature=0.0`) keep call sites clean.

## Step 2 — Define the Protocol (The Interface)

A `Protocol` is Python's way of saying *"any class with this method signature is compatible."* No inheritance required — if it has the right method, it fits.

In [2]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class AIClient(Protocol):
    """Any AI backend must implement this single method."""

    def complete(self, request: AIRequest) -> AIResponse:
        """Send a request and get a response."""
        ...


print("AIClient protocol defined.")
print(f"Required method: complete(request: AIRequest) -> AIResponse")

AIClient protocol defined.
Required method: complete(request: AIRequest) -> AIResponse


**Key points:**
- `@runtime_checkable` lets us use `isinstance()` to verify a backend at runtime.
- The protocol has exactly **one method**: `complete`. This is the seam from the previous lab.
- Any class that implements `complete(AIRequest) -> AIResponse` is a valid `AIClient` — no base class needed.

## Step 3 — Build Two Backends

We'll create a **mock** backend (for testing) and a **simple echo** backend. Both satisfy the `AIClient` protocol without inheriting from it.

In [3]:
class MockAIClient:
    """Always returns a fixed answer — perfect for testing."""

    def __init__(self, fixed_answer: str = "billing"):
        self.fixed_answer = fixed_answer

    def complete(self, request: AIRequest) -> AIResponse:
        return AIResponse(text=self.fixed_answer, model="mock-1.0")


class EchoAIClient:
    """Echoes the prompt back — useful for debugging pipelines."""

    def complete(self, request: AIRequest) -> AIResponse:
        return AIResponse(
            text=f"[echo] {request.prompt}",
            model="echo-1.0",
        )


# Verify both satisfy the protocol at runtime
mock = MockAIClient()
echo = EchoAIClient()

print(f"MockAIClient satisfies AIClient? {isinstance(mock, AIClient)}")
print(f"EchoAIClient satisfies AIClient? {isinstance(echo, AIClient)}")

MockAIClient satisfies AIClient? True
EchoAIClient satisfies AIClient? True


Neither class inherits from `AIClient`. Python's `Protocol` uses **structural subtyping** — if the method signature matches, the contract is satisfied. This is often called *duck typing with guardrails*.

## Step 4 — Dependency Injection in Action

Now let's write the ticket classifier from the previous lab — but this time, the AI backend is **injected** as a parameter. The function has zero vendor imports.

In [4]:
def classify_ticket(ticket_text: str, ai_client: AIClient) -> str:
    """Classify a support ticket. The AI backend is injected."""

    # --- business logic: build the request ---
    categories = ["billing", "technical", "account", "other"]
    request = AIRequest(
        prompt=(
            f"Classify this support ticket into one of {categories}.\n"
            f"Ticket: {ticket_text}\nCategory:"
        ),
        temperature=0.0,
    )

    # --- the seam: one call, no vendor details ---
    response = ai_client.complete(request)

    # --- business logic: validate the answer ---
    category = response.text.strip().lower()
    if category not in categories:
        category = "other"
    return category


# Use the mock backend — no API key, no network
result = classify_ticket("I was charged twice this month", mock)
print(f"Mock backend → {result}")

Mock backend → billing


**What just happened:**
- `classify_ticket` has **zero vendor imports**.
- The AI backend is a parameter (`ai_client`) — that's dependency injection.
- Swapping from mock to echo (or to a real OpenAI backend later) requires changing **one argument**, not editing the function.

## Step 5 — What Doesn't Fit the Protocol

Let's verify that a class **without** the right method is correctly rejected.

In [5]:
class BrokenClient:
    """Missing the complete() method entirely."""

    def generate(self, text: str) -> str:
        return text


broken = BrokenClient()
print(f"BrokenClient satisfies AIClient? {isinstance(broken, AIClient)}")
print()
print("→ The protocol catches the mismatch: no complete() method found.")

BrokenClient satisfies AIClient? False

→ The protocol catches the mismatch: no complete() method found.


## Step 6 — The Full Picture

Let's visualize the architecture we just built.

In [6]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

diagram = """
## Architecture After This Lab

```
┌──────────────────────────────────┐
│  Business Logic                  │
│  classify_ticket(text, ai_client)│
│  • builds AIRequest              │
│  • validates AIResponse          │
│  • zero vendor imports           │
└──────────┬───────────────────────┘
           │  ai_client.complete(request)
           │  (the seam)
    ┌──────┴──────┐
    │  AIClient   │  ← Protocol (interface)
    │  Protocol   │
    └──────┬──────┘
     ┌─────┼─────────┐
     │     │         │
     ▼     ▼         ▼
  Mock   Echo   (Future: OpenAI,
                  Ollama, etc.)
```

**Dependency injection** = the caller decides which backend to pass in.  
**Protocol** = the contract that all backends must satisfy.  
**Dataclasses** = the typed shapes that flow across the seam.
"""

md(diagram)


## Architecture After This Lab

```
┌──────────────────────────────────┐
│  Business Logic                  │
│  classify_ticket(text, ai_client)│
│  • builds AIRequest              │
│  • validates AIResponse          │
│  • zero vendor imports           │
└──────────┬───────────────────────┘
           │  ai_client.complete(request)
           │  (the seam)
    ┌──────┴──────┐
    │  AIClient   │  ← Protocol (interface)
    │  Protocol   │
    └──────┬──────┘
     ┌─────┼─────────┐
     │     │         │
     ▼     ▼         ▼
  Mock   Echo   (Future: OpenAI,
                  Ollama, etc.)
```

**Dependency injection** = the caller decides which backend to pass in.  
**Protocol** = the contract that all backends must satisfy.  
**Dataclasses** = the typed shapes that flow across the seam.


## 🎯 Summary

### Key Takeaways

1. **Protocol as interface** — a Python `Protocol` defines the contract (`complete(AIRequest) -> AIResponse`) that any AI backend must satisfy, without requiring inheritance
2. **Dataclass shapes** — `AIRequest` and `AIResponse` dataclasses make the data flowing across the seam explicit, typed, and self-documenting
3. **Dependency injection** — passing the `ai_client` as a parameter means the business logic never imports or knows about any specific vendor
4. **Instant testability** — a `MockAIClient` that returns a fixed answer lets you test the entire pipeline without an API key or network access